# Goals

Determine how the bug described in https://github.com/IQSS/dataverse.harvard.edu/issues/500 can adversley affect search results of users of Harvard Dataverse and other Dataverse repositories.

# Plan

- Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation and Funding Information Agency fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

- For Harvard Dataverse:
    - Get the list of URLs and names of RORs in the externalvocabularyvalue table

    - Get the list of author affiliations with RORs; the list and count of datasets that have those author affiliations; and if each ROR is in the externalvocabularyvalue table
    - For each author affiliation with a ROR:
        - Search 1: Use the Search API to search for datasets by the name of the author affiliation
        - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
        - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

- For other Dataverse installations:
    - Determine which Dataverse installations have datasets with ROR URLs in their Author Affiliations or Funding Information Agencies
    - For each installation:

        - Get the list of author affiliations with RORs and the list and count of datasets that have those author affiliations (I don't know what's in each installation's externalvocabularyvalue table, so I can't include if each ROR is in the externalvocabularyvalue table)
        - For each author affiliation with a ROR:
            - Search 1: Use the Search API to search for datasets by the name of the author affiliation
            - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
            - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

# Import libraries and set variables

In [ ]:
import csv
import json
import os
from os import listdir
import pandas as pd
from pathlib import Path
import platform
import re
import requests
import sys
import time
from time import sleep
from tqdm import tqdm
tqdmBarFormat = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions
from dataverse_functions import dataverse_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

In [ ]:
# Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation 
# fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

allDataverseJSONExportFilesDirectory = '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/all_json_metadata_files'

# From allDataverseJSONExportFilesDirectory, get the first maxFileCount JSON files whose names include "(latest_version)" and save them to a new directory on my desktop
import os
import shutil

destinationDirectory = '/Users/juliangautier/Desktop/latest_version_json_files'
os.makedirs(destinationDirectory, exist_ok=True)

maxFileCount = 5000

# Get the paths of all files in allDataverseJSONExportFilesDirectory so that the progress bar can show how many files have been checked
allFilePaths = []

for root, dirs, files in os.walk(allDataverseJSONExportFilesDirectory):
    for file in files:
        allFilePaths.append(os.path.join(root, file))

fileCount = len(os.listdir(destinationDirectory))

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=allFilePaths)
for filePath in loopObj:
    loopObj.set_postfix_str(f'{os.path.basename(filePath)} (copied: {fileCount})')
    if "(latest_version)" in os.path.basename(filePath):
        shutil.copy(filePath, destinationDirectory)
        fileCount = len(os.listdir(destinationDirectory))
        if fileCount >= maxFileCount:
            break

# Print count of files 
print(len(os.listdir(destinationDirectory)))

# Get all names, ROR URLs in Author Affiliation and Funding Information Agency fields
Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation and Funding Information Agency fields in datasets published in all known Dataverse installations. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

In [ ]:
# Set the directory of JSON files to parse (destinationDirectory is set in the copy cell above) 
# and the path of the CSV file to save
jsonDirectory = destinationDirectory

In [ ]:
# Parse each JSON file and collect the ROR URL and organization name of each author affiliation that has a ROR URL,
# along with the name of the file that has it
rorRows = []
filesWithRorsCount = 0

def findAuthorAffiliationRors(obj, rows, fileName):
    # Recursively search the metadata for authorAffiliation fields whose value is a ROR URL
    if isinstance(obj, dict):
        if obj.get('typeName') == 'authorAffiliation' and 'ror.org' in str(obj.get('value', '')):
            rows.append({
                'ror_url': obj['value'],
                'organization_name': obj.get('expandedvalue', {}).get('termName'),
                'file_name': fileName})
        for value in obj.values():
            findAuthorAffiliationRors(value, rows, fileName)
    elif isinstance(obj, list):
        for item in obj:
            findAuthorAffiliationRors(item, rows, fileName)

jsonFileNames = [f for f in os.listdir(jsonDirectory) if f.endswith('.json')]

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=jsonFileNames)
for jsonFileName in loopObj:
    loopObj.set_postfix_str(jsonFileName)
    with open(os.path.join(jsonDirectory, jsonFileName), 'r', encoding='utf-8') as f:
        datasetMetadata = json.load(f)

    rowCountBefore = len(rorRows)
    findAuthorAffiliationRors(datasetMetadata, rorRows, jsonFileName)
    if len(rorRows) > rowCountBefore:
        filesWithRorsCount += 1

rorsDf = pd.DataFrame(rorRows, columns=['ror_url', 'organization_name', 'file_name'])

In [ ]:
# Create one row per unique ROR URL. Include as lists the organization names and file names that have each ROR,
# the count of organization names, and the count of files that have each ROR
uniqueRorsDf = (
    rorsDf
    .groupby('ror_url')
    .agg(
        organization_names_from_export=('organization_name', lambda x: sorted(set(x.dropna()))),
        dataset_names=('file_name', lambda x: sorted(set(x))))
    .reset_index()
    .sort_values('ror_url'))
uniqueRorsDf['organization_name_count'] = uniqueRorsDf['organization_names_from_export'].apply(len)
uniqueRorsDf['dataset_count'] = uniqueRorsDf['dataset_names'].apply(len)

uniqueRorsDf.head()

In [ ]:
# For each ROR URL that has no organization name in uniqueRorsDf, get the organization name from the ROR API
# and attach to a new column, organization_names_from_ROR_API, in uniqueRorsDf

rorUrlsWithoutName = uniqueRorsDf.loc[
    uniqueRorsDf['organization_names_from_export'].apply(len) == 0, 'ror_url'].tolist()

# Dictionary of ROR URLs and the one organization name that the ROR API returns for each
rorApiNames = {}
failedRorUrls = []

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=rorUrlsWithoutName)
for rorUrl in loopObj:
    loopObj.set_postfix_str(f'{rorUrl}, failed: {len(failedRorUrls)}')
    try:
        rorInfoDict = common_functions.get_org_data_from_ror(rorUrl)
        rorApiName = rorInfoDict['displayName']
        rorApiNames[rorUrl] = rorApiName
    except requests.exceptions.RequestException:
        failedRorUrls.append(rorUrl)
    sleep(1)

# The new column is empty for ROR URLs that have a name in the export
uniqueRorsDf['organization_names_from_ROR_API'] = uniqueRorsDf['ror_url'].map(rorApiNames)

print(f'ROR URLs without a name in the export: {len(rorUrlsWithoutName)}; failed API requests: {len(failedRorUrls)}')

In [ ]:
# Save the ROR URLs, organization names and dataset names to a CSV file on my desktop
uniqueRorsDf = uniqueRorsDf[[
    'ror_url', 'organization_names_from_export', 'organization_name_count', 'organization_names_from_ROR_API', 'dataset_count', 'dataset_names'
]]

csvFilePath = os.path.join(desktopPath, 'author_affiliation_rors.csv')
uniqueRorsDf.to_csv(csvFilePath, index=False, encoding='utf-8-sig')

print(f'Saved {len(uniqueRorsDf)} unique ROR URLs to {csvFilePath}')
uniqueRorsDf.head()

In [ ]:
# Summarize the data in uniqueRorsDf (the data saved in author_affiliation_rors.csv). 
# Save the summary to a text file on my desktop
summaryFilePath = os.path.join(desktopPath, 'author_affiliation_rors_data_quality_checks.txt')

# ROR URLs that don't match the pattern https://ror.org/ followed by 9 alphanumeric characters
malformedRorUrlCount = (~uniqueRorsDf['ror_url'].str.fullmatch(r'https://ror\.org/[0-9a-z]{9}')).sum()

hasExportName = uniqueRorsDf['organization_name_count'] > 0
hasApiName = uniqueRorsDf['organization_names_from_ROR_API'].notna()

summaryLines = [
    f'Files parsed: {len(jsonFileNames)}',
    f'Files with at least one ROR in an author affiliation: {uniqueRorsDf["dataset_names"].explode().nunique()}',
    f'Unique ROR URLs: {len(uniqueRorsDf)}',
    f'Malformed ROR URLs: {malformedRorUrlCount}',
    f'ROR URLs with a name in the export: {hasExportName.sum()}',
    f'ROR URLs with more than one name in the export: {(uniqueRorsDf["organization_name_count"] > 1).sum()}',
    f'ROR URLs with no name in the export: {(~hasExportName).sum()}',
    f'ROR URLs with no name in the export that got a name from the ROR API: {(~hasExportName & hasApiName).sum()}',
    f'ROR URLs with no name in the export or from the ROR API: {(~hasExportName & ~hasApiName).sum()}',
    f'Failed ROR API requests: {len(failedRorUrls)}']

with open(summaryFilePath, mode='w', encoding='utf-8') as f:
    f.write('\n'.join(summaryLines) + '\n')

In [ ]:
# Load and query the Dataverse JSON exports of the latest version of each dataset. If the export includes ROR URLs in the author affiliation 
# field, save the dataset PID, the ROR URL, and ROR name if it's there

filePath = '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataverse_json_metadata_from_each_known_dataverse_installation_unzipped/CORA._Research_Data_Repository_(RDR)_2026.09.02_11.05.39/Dataverse_JSON_metadata_2026.09.02_11.05.39/doi_10.34810_DATA3521_v1.0(latest_version).json'
with open(filePath, 'r') as f1:
    datasetMetadata = f1.read()
    datasetMetadata = json.loads(datasetMetadata)

citationBlock = datasetMetadata['data']['metadataBlocks']['citation']
for field in citationBlock['fields']:
    if field['typeName'] == 'author':
        authors = field['value']

for author in authors:
    authorAffiliationName = author['authorAffiliation']['expandedvalue']['termName']
    authorAffiliationRor = author['authorAffiliation']['value']

    print(f'{authorAffiliationName} ({authorAffiliationRor})')

# print(json.dumps(authors, indent=4))

authorsDf = pd.read_csv(
    'author(citation)_2026.08.25-2026.09.02.csv',
    # parse_dates=['dataset_publication_date'],
    usecols=lambda x: x in ['dataset_pid', 'dataset_pid_url', 'authorAffiliation'],
    # dtype={'latest_dataset_version_number': str},
    encoding='utf-8-sig'
)

renamedColumns = {'authorAffiliation': 'author_affiliation'}
authorsDf = authorsDf.rename(columns=renamedColumns)

# Create new dataframe with only authors where there's a ROR in the authorAffiliation field, using regex to match the ROR pattern (https://ror.org/ followed by 9 alphanumeric characters)
authorsWithRORAffiliationsDf = authorsDf[authorsDf['author_affiliation'].str.contains(r'https://ror\.org/[0-9a-zA-Z]{9}', regex=True, na=False)]

print(f'Count of rows in authorsWithRORAffiliationsDf: {len(authorsWithRORAffiliationsDf)}')

authorsWithRORAffiliationsDf.info()

Use the dataset PID and my dataset_pids_from_most_known_dataverse_installations.csv file to indicate which installation each dataset PID is in, so that I can group the PIDs by Dataverse installation

In [ ]:
objectColumns = [
    'dataverse_installation_name',
    'dataset_pid',
    'dataset_pid_url']
typeDict = {col: str for col in objectColumns}

datasetsByInstallationsDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataset_pids_from_most_known_dataverse_installations.csv',
    usecols=lambda x: x in ['dataverse_installation_name', 'dataset_pid', 'dataset_pid_url'],
    dtype=typeDict,
    encoding='utf-8-sig'
)

# Measure bug's affect on findability of datasets in Harvard Dataverse

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in Harvard Dataverse

# Measure bug's affect on findability of datasets in other Dataverse installations

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in each Dataverse installation